In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [6]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_23_8_0,0.999988,0.708987,0.999953,0.999973,0.999971,0.000007,0.172758,0.000020,0.000023,0.000021,0.000846,0.002616,1.000005,0.002727,175.785103,268.419666,"Hidden Size=[7, 6], regularizer=0.03, learning..."
1,model_23_8_2,0.999988,0.708987,0.999953,0.999973,0.999971,0.000007,0.172758,0.000020,0.000023,0.000021,0.000846,0.002616,1.000005,0.002727,175.785017,268.419579,"Hidden Size=[7, 6], regularizer=0.03, learning..."
2,model_23_8_1,0.999988,0.708987,0.999953,0.999973,0.999971,0.000007,0.172758,0.000020,0.000023,0.000021,0.000846,0.002616,1.000005,0.002727,175.784991,268.419554,"Hidden Size=[7, 6], regularizer=0.03, learning..."
3,model_23_9_15,0.999988,0.708987,0.999977,0.999994,0.999988,0.000007,0.172758,0.000007,0.000003,0.000005,0.000846,0.002616,1.000005,0.002727,175.784982,268.419544,"Hidden Size=[7, 6], regularizer=0.03, learning..."
4,model_23_9_12,0.999988,0.708987,0.999977,0.999994,0.999988,0.000007,0.172758,0.000007,0.000003,0.000005,0.000846,0.002616,1.000005,0.002727,175.784982,268.419544,"Hidden Size=[7, 6], regularizer=0.03, learning..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1356,model_16_6_9,0.999697,0.644089,0.999932,0.999929,0.999930,0.000180,0.211284,0.000017,0.000090,0.000053,0.004179,0.013414,1.000269,0.013985,119.245721,181.408388,"Hidden Size=[6, 4], regularizer=0.03, learning..."
1361,model_7_8_4,0.999693,0.653897,0.999950,0.999792,0.999853,0.000182,0.205462,0.000031,0.000205,0.000118,0.004243,0.013489,1.000171,0.014064,151.223444,232.888124,"Hidden Size=[6, 6], regularizer=0.03, learning..."
1380,model_10_7_3,0.999677,0.517500,0.999479,0.999696,0.999673,0.000192,0.286433,0.000467,0.000117,0.000292,0.005752,0.013854,1.000127,0.014444,187.116616,290.721062,"Hidden Size=[7, 7], regularizer=0.5, learning_..."
1381,model_14_3_5,0.999676,0.704920,0.999587,0.999785,0.999737,0.000192,0.175172,0.000146,0.000142,0.000144,0.008269,0.013868,1.000288,0.014459,119.112567,181.275234,"Hidden Size=[6, 4], regularizer=0.5, learning_..."
